# 🧪 ADMET Toxicity Prediction with ChemBERTa
## Milestone 2: Multi-Target Classification across All 12 Tox21 Assays

**What's new vs Milestone 1:**
- **Multi-label output:** The head outputs 12 logits — one per target — instead of 2
- **Masked BCE loss:** NaN entries (untested compounds) are excluded from the loss per-sample
- **Per-target class weights:** Each target gets its own positive weight based on its imbalance ratio
- **Multi-task learning:** All 12 targets share the same ChemBERTa encoder — signal from one target helps all others
- **Per-target thresholds:** Optimal decision thresholds are tuned independently for each assay

**Why multi-task learning helps:**  
A compound that damages DNA tends to activate *multiple* stress-response pathways simultaneously.  
Training on all 12 targets at once lets the model learn shared toxic substructures that generalise  
better than 12 separate single-task models.

**Architecture:**
```
SMILES Input
    ↓
ChemBERTa-77M (frozen + LoRA adapters)
    ↓
Classification Head: Linear(600 → 12)
    ↓
12 independent sigmoid probabilities
[NR-AR, NR-AR-LBD, NR-AhR, ..., SR-p53]
```

**Runtime:** GPU recommended (T4 in Colab, or local MPS on Apple Silicon)

## Section 1: Setup

In [ ]:
# Same dependencies as Milestone 1
!pip install -q transformers datasets peft accelerate wandb scikit-learn huggingface_hub python-dotenv

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datetime import datetime

from datasets import load_dataset, Dataset as HFDataset, Sequence, Value, Features
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
import wandb
from huggingface_hub import login

SEED = 42
set_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Configuration

Key differences from Milestone 1:
- No `TARGET_TASK` — we use all 12 targets simultaneously
- `NUM_TARGETS = 12` drives the model head and loss function
- `metric_for_best_model` is now `mean_roc_auc` (average across all 12 targets)

In [ ]:
# ── Model & Data ──────────────────────────────────────────────────────────────
BASE_MODEL   = "DeepChem/ChemBERTa-77M-MTR"
DATASET_NAME = "scikit-fingerprints/MoleculeNet_Tox21"
PROJECT_NAME = "admet-toxicity"
HF_USER      = "mike-malloy"

# All 12 Tox21 targets — fixed order matters for label tensor indexing
TASK_COLUMNS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]
NUM_TARGETS = len(TASK_COLUMNS)   # 12

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1
TARGET_MODULES  = ["query", "value"]

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS                   = 10
BATCH_SIZE               = 32
LEARNING_RATE            = 2e-4
WEIGHT_DECAY             = 0.01
WARMUP_STEPS             = 50
MAX_SEQ_LENGTH           = 128
LOG_STEPS                = 10
EVAL_STEPS               = 50
SAVE_STEPS               = 50
EARLY_STOPPING_PATIENCE  = 4

# ── Derived ───────────────────────────────────────────────────────────────────
RUN_NAME       = f"chemberta-tox21-multitarget-{datetime.now():%Y%m%d_%H%M}"
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

print(f"Model:       {BASE_MODEL}")
print(f"Targets:     {NUM_TARGETS}")
print(f"Run name:    {RUN_NAME}")

### Login to HuggingFace and Weights & Biases

Same as Milestone 1 — reads from `.env` locally or Colab secrets in the Colab UI.

In [ ]:
def _get_secret(key: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key)
    except (ImportError, Exception):
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
        val = os.environ.get(key)
        if not val:
            raise EnvironmentError(
                f"Secret '{key}' not found. "
                f"Add {key}=<value> to a .env file in the project directory."
            )
        return val

hf_token = _get_secret('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

wandb_api_key = _get_secret('WANDB_API_KEY')
os.environ["WANDB_API_KEY"]   = wandb_api_key
os.environ["WANDB_PROJECT"]   = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"]     = "false"
wandb.login()

wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "base_model":     BASE_MODEL,
        "num_targets":    NUM_TARGETS,
        "task":           "multi_label_classification",
        "lora_r":         LORA_R,
        "lora_alpha":     LORA_ALPHA,
        "lora_dropout":   LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "epochs":         EPOCHS,
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "seed":           SEED,
    }
)
print("\u2705 Logged in to HuggingFace and Weights & Biases")

---
## Section 2: Data

### Multi-label vs multi-class

In Milestone 1 the label was a single integer (0 or 1) — a **multi-class** problem.  
Here each compound gets a **vector of 12 floats** — a **multi-label** problem:

```
compound X → [0.0, 0.0, 1.0, None, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
               NR-AR       NR-AhR↑       ...    SR-ARE↑        ...    SR-p53
```

`None` (NaN) means the compound was never tested against that target.  
We use **−1 as a sentinel** for NaN and **mask those entries out of the loss** — they contribute  
no gradient, neither positive nor negative.

In [ ]:
print("Loading Tox21 dataset...")
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)

sample = raw_dataset['train'][0]
SMILES_COL = 'smiles' if 'smiles' in sample else 'SMILES'
print(f"\nSMILES column: '{SMILES_COL}'")

In [ ]:
# ── Label distribution across all 12 targets ──────────────────────────────────
df_raw = raw_dataset['train'].to_pandas()

print(f"{'Target':<20} {'N tested':<10} {'N toxic':<10} {'% toxic':<10} {'pos_weight'}")
print("-" * 62)
for t in TASK_COLUMNS:
    col      = df_raw[t]
    n_tested = col.notna().sum()
    n_pos    = (col == 1).sum()
    n_neg    = n_tested - n_pos
    pct      = 100 * n_pos / n_tested if n_tested > 0 else 0
    pw       = n_neg / n_pos if n_pos > 0 else float('inf')
    print(f"{t:<20} {n_tested:<10} {n_pos:<10} {pct:<10.1f}% {pw:.1f}x")

In [ ]:
# ── Preprocess: build label matrix and create train/val/test splits ────────────
#
# Label encoding:
#   1.0  = toxic (positive)
#   0.0  = non-toxic (negative)
#  -1.0  = not tested (masked out of loss)

df = df_raw.copy()
df = df[df[TASK_COLUMNS].notna().any(axis=1)]   # drop rows with no labels at all

label_matrix = df[TASK_COLUMNS].fillna(-1).values.astype(np.float32)
smiles_arr   = df[SMILES_COL].values

processed_df = pd.DataFrame({
    'smiles': smiles_arr,
    'labels': [row.tolist() for row in label_matrix],
})

# Stratify on "has at least one toxic label" as a proxy for multi-label balance
stratify_col = (label_matrix.max(axis=1) > 0).astype(int)

train_df, temp_df, _, strat_temp = train_test_split(
    processed_df, stratify_col,
    test_size=0.30, random_state=SEED, stratify=stratify_col,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=strat_temp,
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

for name, df_ in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    labels_arr = np.array(df_['labels'].tolist())
    any_toxic  = (labels_arr.max(axis=1) > 0).mean()
    print(f"{name:<6}: {len(df_):>5} compounds | {any_toxic:.1%} have \u2265 1 toxic label")

---
## Section 3: Tokenization

Tokenization is identical to Milestone 1 — ChemBERTa tokenises SMILES the same way  
regardless of whether we're doing single- or multi-target classification.  
The only change is the label format: instead of a scalar int, each sample now carries  
a list of 12 floats.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"Vocabulary size: {tokenizer.vocab_size}")

def tokenize_batch(examples):
    return tokenizer(
        examples['smiles'],
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

# Declare explicit feature types so HuggingFace knows labels are a fixed-length
# float sequence — this ensures correct tensor shapes when set_format is called.
features = Features({
    'smiles': Value('string'),
    'labels': Sequence(Value('float32'), length=NUM_TARGETS),
})

def make_dataset(df):
    ds = HFDataset.from_dict(
        {'smiles': df['smiles'].tolist(), 'labels': df['labels'].tolist()},
        features=features,
    )
    ds = ds.map(tokenize_batch, batched=True)
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

print("Tokenized datasets:")
print(f"  Train: {train_dataset}")
sample = train_dataset[0]
print(f"\ninput_ids shape: {sample['input_ids'].shape}")
print(f"labels shape:    {sample['labels'].shape}  (one float per target)")
print(f"labels sample:   {sample['labels']}")

---
## Section 4: Model — ChemBERTa with a 12-output Head

The only change from Milestone 1 is `num_labels=12`.  
`AutoModelForSequenceClassification` replaces the regression head with  
`Linear(600 → 12)` — 12 independent logits, one per Tox21 target.  
We set `problem_type="multi_label_classification"` to document intent,  
though our custom trainer handles the loss directly.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_TARGETS,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,
    trust_remote_code=True,
)

total = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters:  {total:,}")
print(f"Memory footprint:  {base_model.get_memory_footprint() / 1e6:.1f} MB")
print("\nTop-level modules:")
for name, module in base_model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {type(module).__name__:<35} {n:>10,} params")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

---
## Section 5: Training

### Two changes from Milestone 1

**1. Masked BCE loss instead of cross-entropy**

Cross-entropy works on a single label. For 12 independent binary predictions we need  
`BCEWithLogitsLoss`, applied element-wise:

```
loss = BCEWithLogitsLoss(logits, targets, pos_weight=pw) * mask
```

The `mask` zeros out any position where the label is −1 (untested), so those entries  
contribute no gradient in either direction.

**2. Per-target positive weights**

Each target has a different imbalance ratio (NR-PPAR-gamma is 32x imbalanced; SR-ARE is only 5x).  
We compute `pos_weight[i] = n_negative[i] / n_positive[i]` for each target independently,  
then pass the full weight vector to `BCEWithLogitsLoss`.

In [ ]:
# ── Per-target positive weights ───────────────────────────────────────────────
labels_train = np.array(train_df['labels'].tolist())   # [N, 12]

pos_weights_list = []
print(f"{'Target':<20} {'N pos':<8} {'N neg':<8} {'pos_weight'}")
print("-" * 50)
for i, task in enumerate(TASK_COLUMNS):
    col     = labels_train[:, i]
    tested  = col[col != -1]
    n_pos   = (tested == 1).sum()
    n_neg   = (tested == 0).sum()
    pw      = n_neg / n_pos if n_pos > 0 else 1.0
    pos_weights_list.append(pw)
    print(f"{task:<20} {n_pos:<8} {n_neg:<8} {pw:.1f}x")

pos_weights_tensor = torch.tensor(pos_weights_list, dtype=torch.float32)
print(f"\nWeight tensor shape: {pos_weights_tensor.shape}")

In [ ]:
# ── Multi-target evaluation metrics ──────────────────────────────────────────
# Reports ROC-AUC per target and mean ROC-AUC (primary metric).
# Metric names use underscores (HuggingFace Trainer rejects hyphens in names).

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(
        torch.tensor(logits, dtype=torch.float32)
    ).numpy()   # [N, 12]

    aucs = {}
    for i, task in enumerate(TASK_COLUMNS):
        mask       = labels[:, i] != -1
        if mask.sum() < 2:
            continue
        task_labels = labels[mask, i]
        task_probs  = probs[mask, i]
        try:
            auc = roc_auc_score(task_labels, task_probs)
            key = "auc_" + task.replace("-", "_")
            aucs[key] = round(float(auc), 4)
        except ValueError:
            pass

    mean_auc = round(float(np.mean(list(aucs.values()))), 4) if aucs else 0.0
    return {"mean_roc_auc": mean_auc, **aucs}

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
use_mps  = not torch.cuda.is_available() and torch.backends.mps.is_available()

if use_bf16:   precision = "bf16 (CUDA)"
elif use_fp16: precision = "fp16 (CUDA)"
elif use_mps:  precision = "fp32 (MPS / Apple Silicon)"
else:          precision = "fp32 (CPU)"
print(f"Precision/device: {precision}")

training_args = TrainingArguments(
    output_dir=RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,
    fp16=use_fp16,
    bf16=use_bf16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="mean_roc_auc",
    greater_is_better=True,
    logging_steps=LOG_STEPS,
    report_to="wandb",
    run_name=RUN_NAME,
    dataloader_pin_memory=False,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    hub_strategy="every_save",
    seed=SEED,
)

print(f"Steps per epoch: {math.ceil(len(train_dataset) / BATCH_SIZE)}")
print(f"Total steps:     {math.ceil(len(train_dataset) / BATCH_SIZE) * EPOCHS}")

In [ ]:
# ── Multi-target Trainer with masked BCE loss ─────────────────────────────────
#
# Key differences from Milestone 1's WeightedLossTrainer:
#   - BCEWithLogitsLoss (not cross-entropy) for independent binary outputs
#   - mask zeros out untested entries (-1) per sample per target
#   - pos_weight is a vector of 12 values, one per target

class MultiTargetTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels").float()   # [batch, 12]
        outputs = model(**inputs)
        logits  = outputs.logits                 # [batch, 12]

        # Mask: 1 where tested, 0 where label == -1
        mask    = (labels != -1).float()
        targets = labels.clamp(min=0)            # -1 → 0 (will be zeroed by mask)

        loss_per_element = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=pos_weights_tensor.to(logits.device),
            reduction='none',
        )

        # Average only over tested (mask == 1) entries
        loss = (loss_per_element * mask).sum() / mask.sum().clamp(min=1)
        return (loss, outputs) if return_outputs else loss


trainer = MultiTargetTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f"Training on   {len(train_dataset):,} samples")
print(f"Validating on {len(val_dataset):,} samples")
print(f"\nStarting training... \U0001f680")
train_result = trainer.train()

---
## Section 6: Evaluation

We evaluate two ways:
1. **ROC-AUC per target** — measures ranking quality regardless of threshold
2. **Per-target threshold tuning** — finds the decision boundary that maximises F1 on the validation set, then reports recall/precision on the test set

In [ ]:
# ── ROC-AUC per target on the test set ───────────────────────────────────────
test_preds  = trainer.predict(test_dataset)
test_logits = test_preds.predictions           # [N, 12]
test_labels = test_preds.label_ids             # [N, 12]
test_probs  = torch.sigmoid(
    torch.tensor(test_logits, dtype=torch.float32)
).numpy()

print(f"{'Target':<20} {'ROC-AUC':<10} {'N tested':<10} {'% toxic'}")
print("-" * 52)
aucs = {}
for i, task in enumerate(TASK_COLUMNS):
    mask        = test_labels[:, i] != -1
    task_labels = test_labels[mask, i]
    task_probs_ = test_probs[mask, i]
    n_tested    = mask.sum()
    pct_toxic   = 100 * task_labels.mean()
    try:
        auc = roc_auc_score(task_labels, task_probs_)
        aucs[task] = auc
        print(f"{task:<20} {auc:<10.4f} {n_tested:<10} {pct_toxic:.1f}%")
    except ValueError:
        print(f"{task:<20} {'N/A':<10} {n_tested:<10} {pct_toxic:.1f}%")

print(f"\n{'Mean ROC-AUC':<20} {np.mean(list(aucs.values())):.4f}")

In [ ]:
# ── Per-target threshold tuning on validation set ─────────────────────────────
val_preds  = trainer.predict(val_dataset)
val_probs  = torch.sigmoid(
    torch.tensor(val_preds.predictions, dtype=torch.float32)
).numpy()
val_labels = val_preds.label_ids

best_thresholds = {}

print(f"{'Target':<20} {'Threshold':<12} {'Val F1':<10} {'Recall':<10} {'Precision'}")
print("-" * 65)
for i, task in enumerate(TASK_COLUMNS):
    mask      = val_labels[:, i] != -1
    tl        = val_labels[mask, i]
    tp        = val_probs[mask, i]

    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.05, 0.70, 0.05):
        preds_t = (tp >= t).astype(int)
        f = f1_score(tl, preds_t, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t

    best_thresholds[task] = best_t
    preds_best = (tp >= best_t).astype(int)
    from sklearn.metrics import recall_score, precision_score
    rec  = recall_score(tl, preds_best, zero_division=0)
    prec = precision_score(tl, preds_best, zero_division=0)
    print(f"{task:<20} {best_t:<12.2f} {best_f:<10.3f} {rec:<10.3f} {prec:.3f}")

print(f"\nThresholds saved to `best_thresholds` dict.")

In [ ]:
# ── Per-target test recall summary ────────────────────────────────────────────
from sklearn.metrics import recall_score, precision_score, confusion_matrix

print(f"{'Target':<20} {'Recall':<10} {'Precision':<12} {'TP':<6} {'FN':<6} {'FP'}")
print("-" * 62)
for i, task in enumerate(TASK_COLUMNS):
    mask      = test_labels[:, i] != -1
    tl        = test_labels[mask, i]
    tp        = test_probs[mask, i]
    preds     = (tp >= best_thresholds[task]).astype(int)

    rec  = recall_score(tl, preds, zero_division=0)
    prec = precision_score(tl, preds, zero_division=0)
    cm   = confusion_matrix(tl, preds, labels=[0, 1])
    fn   = cm[1, 0] if cm.shape == (2, 2) else 0
    tp_  = cm[1, 1] if cm.shape == (2, 2) else 0
    fp   = cm[0, 1] if cm.shape == (2, 2) else 0
    print(f"{task:<20} {rec:<10.3f} {prec:<12.3f} {tp_:<6} {fn:<6} {fp}")

---
## Section 7: Inference

`predict_toxicity` now returns a prediction for all 12 targets at once,  
using the per-target thresholds tuned on the validation set.

In [ ]:
def predict_toxicity(smiles: str, thresholds: dict = None) -> dict:
    """
    Predict toxicity across all 12 Tox21 targets for a single SMILES string.

    Args:
        smiles:     A valid SMILES string.
        thresholds: Dict mapping target name → decision threshold.
                    Defaults to 0.5 for all targets; use `best_thresholds`
                    from the evaluation cell for calibrated predictions.

    Returns:
        Dict mapping target name → {probability, prediction}
    """
    if thresholds is None:
        thresholds = {t: 0.5 for t in TASK_COLUMNS}

    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits[0]).cpu().numpy()

    return {
        task: {
            "probability": round(float(probs[i]), 4),
            "prediction":  "TOXIC" if probs[i] >= thresholds[task] else "NON-TOXIC",
        }
        for i, task in enumerate(TASK_COLUMNS)
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
molecules = {
    "Aspirin":           "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":          "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
    "Cisplatin (chemo)": "[Pt](Cl)(Cl)(N)N",
    "Benzene":           "c1ccccc1",
}

for mol_name, smi in molecules.items():
    results = predict_toxicity(smi, thresholds=best_thresholds)
    toxic_targets = [t for t, r in results.items() if r['prediction'] == 'TOXIC']
    print(f"\n{mol_name} ({smi})")
    print(f"  Flagged targets: {toxic_targets if toxic_targets else 'none'}")
    print(f"  {'Target':<20} {'P(toxic)':<12} {'Prediction'}")
    print(f"  {'-'*44}")
    for task, r in results.items():
        marker = " <--" if r['prediction'] == 'TOXIC' else ""
        print(f"  {task:<20} {r['probability']:<12.4f} {r['prediction']}{marker}")

---
## Section 8: Save, Push to Hub, and Log Metadata

In [ ]:
trainer.save_model(RUN_NAME)
tokenizer.save_pretrained(RUN_NAME)
print(f"\u2705 Model saved locally to: {RUN_NAME}/")

trainer.push_to_hub(
    commit_message=f"Multi-target ChemBERTa-77M on all 12 Tox21 targets "
                   f"| mean ROC-AUC: {np.mean(list(aucs.values())):.4f}"
)
print(f"\u2705 Model pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

metadata = {
    "experiment": {
        "run_name":    RUN_NAME,
        "timestamp":   datetime.now().isoformat(),
        "hub_model":   HUB_MODEL_NAME,
        "milestone":   2,
    },
    "data": {
        "dataset":        DATASET_NAME,
        "targets":        TASK_COLUMNS,
        "num_targets":    NUM_TARGETS,
        "train_samples":  len(train_dataset),
        "val_samples":    len(val_dataset),
        "test_samples":   len(test_dataset),
        "seed":           SEED,
    },
    "model": {
        "base_model":     BASE_MODEL,
        "lora_r":         LORA_R,
        "lora_alpha":     LORA_ALPHA,
        "lora_dropout":   LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
    },
    "training": {
        "epochs":         EPOCHS,
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "weight_decay":   WEIGHT_DECAY,
        "loss":           "masked_bce_with_per_target_pos_weights",
    },
    "results": {
        "per_target_auc":  {t: round(v, 4) for t, v in aucs.items()},
        "mean_roc_auc":    round(np.mean(list(aucs.values())), 4),
        "thresholds":      best_thresholds,
    },
}

with open(f"{RUN_NAME}/experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\n\u2705 Metadata saved")
print(json.dumps(metadata["results"], indent=2))

In [ ]:
wandb.finish()
print("\u2705 Training complete!")
print(f"   Mean ROC-AUC:    {np.mean(list(aucs.values())):.4f}")
print(f"   WandB dashboard: https://wandb.ai/{PROJECT_NAME}")
print(f"   HuggingFace Hub: https://huggingface.co/{HUB_MODEL_NAME}")

---
## What's Next?

With multi-target classification working, the natural next milestones are:

**Milestone 3 — Explainability:**  
Use attention weights or integrated gradients to highlight which atoms in a SMILES  
string drive each toxicity prediction. Turns a probability score into a chemical insight.

**Milestone 4 — Molecular similarity retrieval:**  
Build a FAISS index over training set embeddings. For any query molecule, retrieve  
the most structurally similar training examples — useful for explaining predictions by analogy.

**Milestone 5 — Streamlit UI:**  
A web app where users draw molecules or paste SMILES and see all 12 toxicity predictions  
with confidence scores, highlighted atoms, and similar known compounds.

---
*Project: ADMET Toxicity Prediction with ChemBERTa | Milestone 2*